In [1]:
library(Rcpp)
library(RcppEigen)
library(RcppDist)
library(RcppArmadillo)
library(mvtnorm)
library(dbarts)
sourceCpp("FirstModel.cpp")

Warning message:
"package 'Rcpp' was built under R version 4.3.3"
Warning message:
"package 'RcppEigen' was built under R version 4.3.3"
Warning message:
"package 'RcppDist' was built under R version 4.3.3"
Registered S3 methods overwritten by 'RcppArmadillo':
  method               from     
  predict.fastLm       RcppEigen
  print.fastLm         RcppEigen
  summary.fastLm       RcppEigen
  print.summary.fastLm RcppEigen


Attaching package: 'RcppArmadillo'


The following objects are masked from 'package:RcppEigen':

    fastLm, fastLmPure


Warning message:
"package 'mvtnorm' was built under R version 4.3.3"


In [2]:
library(progress)

Warning message:
"package 'progress' was built under R version 4.3.3"


# DGP_3

In [3]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

num_gfr<-200

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
results_matrix <- matrix(NA, nrow = num_simulations, ncol = 30)
colnames(results_matrix) <- c("mvbcf_1k_pehe1", "mvbcf_1k_pehe2","mvbcf_0.5k_pehe1", "mvbcf_0.5k_pehe2","mvbcf_0.25k_pehe1", "mvbcf_0.25k_pehe2","mvbcf_0.1k_pehe1", "mvbcf_0.1k_pehe2",
                             "mvbcf_0.05k_pehe1", "mvbcf_0.05k_pehe2",
                             "mvbcf_1k_tau_951", "mvbcf_1k_tau_952","mvbcf_0.5k_tau_951", "mvbcf_0.5k_tau_952","mvbcf_0.25k_tau_951", "mvbcf_0.25k_tau_952","mvbcf_0.1k_tau_951", "mvbcf_0.1k_tau_952",
                             "mvbcf_0.05k_tau_951", "mvbcf_0.05k_tau_952", "mvbcf_1k_tau_951w", "mvbcf_1k_tau_952w","mvbcf_0.5k_tau_951w", "mvbcf_0.5k_tau_952w","mvbcf_0.25k_tau_951w", "mvbcf_0.25k_tau_952w","mvbcf_0.1k_tau_951w", "mvbcf_0.1k_tau_952w",
                             "mvbcf_0.05k_tau_951w", "mvbcf_0.05k_tau_952w")

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-runif(n)
X2<-runif(n)
X3<-runif(n)
X4<-runif(n)
X5<-runif(n)
X6<-rbinom(n, 1, 0.5)
X7<-rbinom(n, 1, 0.5)
X8<-rbinom(n, 1, 0.5)
X9<-sample(c(0, 1, 2, 3, 4), n, replace=T)
X10<-sample(c(0, 1, 2, 3, 4), n, replace=T)

X<-cbind(X1, X2, X3, X4, X5, X6, X7, X8, X9, X10)

Mu1<-(11*sin(pi*X4*X5)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
Mu2<-(9*sin(pi*X1*X2)+22*(X3-0.5)^2+14*X4+8*X6+X9)*10+300

Tau1<-(2*X4+2*X2)*10
Tau2<-(1*X3+3*X5)*10

true_propensity<-X4

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-runif(n_test)
X2_test<-runif(n_test)
X3_test<-runif(n_test)
X4_test<-runif(n_test)
X5_test<-runif(n_test)
X6_test<-rbinom(n_test, 1, 0.5)
X7_test<-rbinom(n_test, 1, 0.5)
X8_test<-rbinom(n_test, 1, 0.5)
X9_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)
X10_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)

X_test<-cbind(X1_test, X2_test, X3_test, X4_test, X5_test, X6_test, X7_test, X8_test, X9_test, X10_test)

Mu1_test<-(11*sin(pi*X4_test*X5_test)+18*(X3_test-0.5)^2+10*X4_test+12*X6_test+X9_test)*10+300
Mu2_test<-(9*sin(pi*X1_test*X2_test)+22*(X3_test-0.5)^2+14*X4_test+8*X6_test+X9_test)*10+300

Tau1_test<-(2*X4_test+2*X2_test)*10
Tau2_test<-(1*X3_test+3*X5_test)*10

true_propensity_test<-X4_test

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-1000
n_burn<-500

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

mvbcf_1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_1k_tau_preds1<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_1k_ate1<-mean(mvbcf_1k_tau_preds1)
mvbcf_1k_tau_preds2<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_1k_ate2<-mean(mvbcf_1k_tau_preds2)

mvbcf_1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_1k_tau_preds1)^2))
mvbcf_1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_1k_tau_951<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_1k_tau_951w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_tau_952<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_1k_tau_952w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-500
n_burn<-250

mvbcf_0.5k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.5k_tau_preds1<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.5k_ate1<-mean(mvbcf_0.5k_tau_preds1)
mvbcf_0.5k_tau_preds2<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.5k_ate2<-mean(mvbcf_0.5k_tau_preds2)

mvbcf_0.5k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.5k_tau_preds1)^2))
mvbcf_0.5k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.5k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.5k_tau_951<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.5k_tau_951w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_tau_952<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.5k_tau_952w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-250
n_burn<-125

mvbcf_0.25k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.25k_tau_preds1<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.25k_ate1<-mean(mvbcf_0.25k_tau_preds1)
mvbcf_0.25k_tau_preds2<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.25k_ate2<-mean(mvbcf_0.25k_tau_preds2)

mvbcf_0.25k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.25k_tau_preds1)^2))
mvbcf_0.25k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.25k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.25k_tau_951<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.25k_tau_951w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_tau_952<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.25k_tau_952w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-100
n_burn<-50

mvbcf_0.1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.1k_tau_preds1<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.1k_ate1<-mean(mvbcf_0.1k_tau_preds1)
mvbcf_0.1k_tau_preds2<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.1k_ate2<-mean(mvbcf_0.1k_tau_preds2)

mvbcf_0.1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.1k_tau_preds1)^2))
mvbcf_0.1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.1k_tau_951<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.1k_tau_951w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_tau_952<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.1k_tau_952w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-25

mvbcf_0.05k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.05k_tau_preds1<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.05k_ate1<-mean(mvbcf_0.05k_tau_preds1)
mvbcf_0.05k_tau_preds2<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.05k_ate2<-mean(mvbcf_0.05k_tau_preds2)

mvbcf_0.05k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.05k_tau_preds1)^2))
mvbcf_0.05k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.05k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.05k_tau_951<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.05k_tau_951w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_tau_952<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.05k_tau_952w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


# Store the results in the matrix
  results_matrix[i, ] <- c(mvbcf_1k_pehe1, mvbcf_1k_pehe2, mvbcf_0.5k_pehe1, mvbcf_0.5k_pehe2, mvbcf_0.25k_pehe1, mvbcf_0.25k_pehe2, mvbcf_0.1k_pehe1, mvbcf_0.1k_pehe2,
                             mvbcf_0.05k_pehe1, mvbcf_0.05k_pehe2,
                             mvbcf_1k_tau_951, mvbcf_1k_tau_952, mvbcf_0.5k_tau_951, mvbcf_0.5k_tau_952, mvbcf_0.25k_tau_951, mvbcf_0.25k_tau_952, mvbcf_0.1k_tau_951, mvbcf_0.1k_tau_952,
                             mvbcf_0.05k_tau_951, mvbcf_0.05k_tau_952, mvbcf_1k_tau_951w, mvbcf_1k_tau_952w, mvbcf_0.5k_tau_951w, mvbcf_0.5k_tau_952w, mvbcf_0.25k_tau_951w, mvbcf_0.25k_tau_952w, mvbcf_0.1k_tau_951w, mvbcf_0.1k_tau_952w,
                             mvbcf_0.05k_tau_951w, mvbcf_0.05k_tau_952w)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "simulation_results_DGP3.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to simulation_results_DGP3.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 44180 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 25411 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 17403 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12831 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 10742 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 41628 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 32016 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 17639 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13167 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11531 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


...

Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 42804 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26691 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 18132 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13619 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12096 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 43161 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26766 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 18609 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13946 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12116 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 43820 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26520 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 18628 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14264 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12886 ms
Simulation completed and results saved to simulation_results_DGP3.csv


1m and 7.5s per iteration

1 hour and 53 minutes per 100 replications

In [4]:
print(results_matrix)

       mvbcf_1k_pehe1 mvbcf_1k_pehe2 mvbcf_0.5k_pehe1 mvbcf_0.5k_pehe2
  [1,]       7.996597       9.929800         6.811232        10.215848
  [2,]       9.460930       6.982867        12.167122         5.231118
  [3,]      10.152937       7.509077         9.443587         6.856006
  [4,]       9.926975       7.860725         9.781514         8.733766
  [5,]       9.707443       7.239179        11.181687         7.972177
  [6,]       8.118558       8.409299         8.990579         6.277454
  [7,]       5.599824       9.149341         5.070072         8.530052
  [8,]       7.260996      15.123500         6.254513        13.985207
  [9,]      10.335196       4.729209        11.587456         5.282434
 [10,]      10.446221      12.402751         9.877275        12.008523
 [11,]      10.899345      10.993478        10.125525        11.171077
 [12,]       7.706535      10.151688         8.037407         9.190809
 [13,]       6.482417      17.500189         7.654090        18.472403
 [14,]